<a href="https://colab.research.google.com/github/andresecha/Methodologie-de-la-recherche-Numerique/blob/main/Speech-to-text/Whisper/Metodological_guide_Whisper_audio_Transcripcion_ARIANE_FR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Comment transformer les sources sonores en textes exploitables pour la recherche ?

## Guide méthodologique pour la transcription automatique de l'audio avec Whisper *(OpenAI)*

**Echavarría, Andrés.**

**Groupe d'intérêt « Acquisition assitée de textes »** · Consortium Huma-Num **ARIANE**

---

### Énoncé de la problématique

Une part considérable de la production intellectuelle, patrimoniale et testimoniale des sciences humaines et sociales n'existe que **sous forme orale** : entretiens de terrain, communications scientifiques, conférences enregistrées, émissions radiophoniques, témoignages audiovisuels, fonds sonores conservés dans les archives et les bibliothèques. Ces matériaux — souvent uniques et irremplaçables — constituent des sources primaires d'une valeur inestimable, mais ils demeurent largement **inaccessibles à la recherche plein texte, à la citation et à l'analyse textuelle** tant qu'ils n'ont pas fait l'objet d'une transcription.

La transcription manuelle est lente, coûteuse et difficilement extensible à de grands corpus. Se pose alors une **question de recherche opérationnelle** :

> *Dans quelle mesure les outils actuels de reconnaissance automatique de la parole (RAP / ASR) permettent-ils d'obtenir des transcriptions fiables de sources orales en français, en espagnol et dans les autres langues de travail de nos communautés de recherche ? Quels sont les paramètres, les limites et les bonnes pratiques à maîtriser pour intégrer ces outils dans nos chaînes de traitement ?*
---
### Ce que propose ce carnet Jupyter

Ce carnet Jupyter pour Google Colab constitue un **guide pratique et raisonné** pour l'utilisation de [**Whisper**](https://github.com/openai/whisper), le modèle de reconnaissance automatique de la parole développé par OpenAI et publié en code ouvert (licence MIT). Whisper a été entraîné sur plus de **680 000 heures** d'audio multilingue étiqueté, ce qui lui confère trois capacités fondamentales pour la recherche :

| Capacité | Description | Intérêt pour la recherche |
|----------|-------------|---------------------------|
| **Transcription** | Convertit la parole en texte dans la langue d'origine | Produire des versions textuelles d'entretiens, de conférences ou de témoignages oraux |
| **Traduction** | Traduit l'audio de n'importe quelle langue vers l'anglais | Faciliter l'accès préliminaire à des sources dans des langues que le chercheur ne maîtrise pas |
| **Détection de la langue** | Identifie automatiquement la langue parlée | Classer des fonds sonores multilingues sans écoute préalable |
---
### Architecture du guide

Le carnet est organisé en **neuf sections** qui suivent l'intégralité de la chaîne de traitement d'une transcription :

1. **Configuration de l'environnement** — Activer le GPU et vérifier les ressources de calcul disponibles.
2. **Installation des dépendances** — Installer Whisper, FFmpeg et yt-dlp.
3. **Modèles disponibles** — Comparer les six tailles de modèle (de `tiny` à `large` et `turbo`) selon le rapport qualité / vitesse / mémoire.
4. **Montage de Google Drive** — Connecter l'espace de stockage personnel pour lire et enregistrer des fichiers.
5. **Obtention de l'audio** — Trois voies : téléversement d'un fichier local, utilisation d'un fichier présent sur Drive, téléchargement de l'audio d'une vidéo YouTube.
6. **Vérification et écoute de l'audio** — Contrôler la durée, le format et la qualité avant de lancer le processus.
7. **Transcription avec Whisper** — Deux méthodes (ligne de commandes et script Python), accompagnées d'exemples pour la transcription de base, la transcription rapide, l'utilisation d'un vocabulaire spécialisé, la traduction vers l'anglais et la gestion des audios problématiques.
8. **Exploration et téléchargement des résultats** — Formats de sortie (`.txt`, `.srt`, `.vtt`, `.tsv`, `.json`) et modalités de lecture ou d'exportation.
9. **Conseils et résolution de problèmes** — Bonnes pratiques face au bruit, aux hallucinations, aux enregistrements longs ou aux erreurs de mémoire.
---
### Conditions d'utilisation

- **Environnement recommandé** : Google Colab avec GPU (T4 ou supérieur).
- **Coût** : L'utilisation de Whisper est gratuite (modèle ouvert). Seule la quota GPU de Colab est consommée.
- **Formats d'entrée** : `.mp3`, `.wav`, `.flac`, `.ogg`, `.m4a`, `.wma`, `.aac`, ainsi que les fichiers vidéo (`.mp4`, `.mkv`, `.avi`, etc.).
- **Principales langues prises en charge** : français, espagnol, anglais, portugais, allemand, italien, catalan, occitan, breton, basque, entre [de nombreuses autres](https://github.com/openai/whisper#available-models-and-languages).

### **Pourquoi utiliser Google Colab ?**
Les modèles les plus grands de Whisper (comme `large` ou `large-v3`) nécessitent environ **10 Go de VRAM** (mémoire de la carte graphique). La plupart des ordinateurs personnels ne disposent pas d'un GPU suffisant, mais Colab nous « ~~prête gratuitement~~ » un **GPU NVIDIA T4 avec 16 Go de VRAM**.

> ⚠️ **Note sur le signe `!`** : dans les carnets Colab, les lignes commençant par `!` sont exécutées comme des **commandes de terminal** (*shell*), et non comme du code Python. C'est l'équivalent de taper une commande dans un terminal/console.
---
---
### Comment citer ce guide

> Echavarría, Andrés. *Guide méthodologique pour la transcription automatique de l'audio avec Whisper (OpenAI)*. Groupe d'intérêt « Acquisition assitée de textes », Consortium Huma-Num ARIANE. [Carnet Jupyter Google Colab], version 1.0, mars 2026.
---
---

> 💡 **Note** : Ce guide est un *document évolutif*. Nous vous invitons à signaler toute erreur, à proposer des améliorations ou à suggérer de nouveaux exemples d'usage par l'intermédiaire du groupe d'intérêt.

---
# ⚙️ 1. Configuration de l'environnement d'exécution

Avant d'exécuter toute cellule, il faut activer le **GPU** dans Colab :

1. Dans le menu supérieur, aller à **Environnement d'exécution → Modifier le type d'environnement d'exécution**.
2. Dans la fenêtre qui s'ouvre, sélectionner **GPU** comme accélérateur matériel (généralement un **T4**).
3. Cliquer sur **Enregistrer**.

Après l'enregistrement, l'environnement redémarre. C'est tout à fait normal.

### Vérifier le GPU assigné
Exécute la cellule suivante pour confirmer qu'un GPU est disponible :

In [ ]:
# Vérifier le GPU assigné par Colab
!nvidia-smi

# Vérification également via PyTorch
import torch
print("\n" + "="*50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU disponible : {gpu_name}")
    print(f"   Mémoire totale : {gpu_mem:.1f} Go")
else:
    print("❌ Aucun GPU détecté. Vérifiez la configuration de l'environnement.")
print("="*50)

Fri Mar 13 15:57:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

---
# 📦 2. Installation des dépendances

Nous avons besoin d'installer :
- **openai-whisper** : le modèle de transcription.
- **ffmpeg** : outil pour traiter les fichiers audio/vidéo (souvent préinstallé dans Colab, mais on s'en assure).
- **yt-dlp** : pour télécharger l'audio depuis YouTube ou d'autres plateformes vidéo.

> 💡 L'option `-qqq` réduit au silence la sortie de l'installation pour ne pas surcharger l'écran de messages.

In [ ]:
# 1) S'assurer que ffmpeg est installé
!apt-get update -qq && apt-get install -y -qq ffmpeg

# 2) Installer Whisper depuis le dépôt officiel d'OpenAI
#    Garantit d'avoir la dernière version disponible
!pip install -U openai-whisper -qqq

# 3) Installer yt-dlp pour télécharger l'audio depuis YouTube
!pip install -U yt-dlp -qqq

print("\n✅ Toutes les dépendances ont été installées correctement.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 49.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.4/182.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.7 MB/s eta 0:00:00

✅ Toutes les dépendances ont été installées correctement.


---
# 🧠 3. Modèles disponibles dans Whisper

Whisper propose **six tailles de modèle** (plus des variantes). Plus le modèle est grand, meilleure est la qualité de transcription — mais il est plus lent et consomme plus de VRAM.

| Modèle | Paramètres | VRAM nécessaire | Vitesse relative | Multilingue | Anglais seul |
|--------|-----------|-----------------|-----------------|-------------|---------------|
| `tiny` | 39 M | ~1 Go | ~10× | ✅ `tiny` | ✅ `tiny.en` |
| `base` | 74 M | ~1 Go | ~7× | ✅ `base` | ✅ `base.en` |
| `small` | 244 M | ~2 Go | ~4× | ✅ `small` | ✅ `small.en` |
| `medium` | 769 M | ~5 Go | ~2× | ✅ `medium` | ✅ `medium.en` |
| `large` | 1 550 M | ~10 Go | 1× | ✅ `large` | ❌ |
| **`turbo`** | **809 M** | **~6 Go** | **~8×** | ✅ `turbo` | ❌ |

### Lequel choisir ?

- **Pour une transcription rapide** (brouillons, notes) : `small` ou `turbo`.
- **Pour la meilleure qualité possible** : `large` (équivalent à `large-v3`, la dernière version).
- **Pour la traduction en anglais** : `medium` ou `large` (⚠️ `turbo` **n'est pas** entraîné pour traduire).
- **Anglais uniquement** : les modèles `.en` (ex. `small.en`) sont légèrement meilleurs pour l'anglais pur.
- **Sur le GPU T4 gratuit de Colab (16 Go de VRAM)** : tous les modèles rentrent sans problème.

> 💡 **Recommandation LA RANA** : pour des audios complexes ou en plusieurs langues, utiliser **`large`**. Pour des audios longs mais simples, **`turbo`** est un bon compromis entre vitesse et qualité.

---
# 📂 4. Monter Google Drive (facultatif)

Si tes fichiers audio se trouvent sur Google Drive, ou si tu souhaites y enregistrer les résultats, exécute la cellule suivante. Colab te demandera d'autoriser l'accès à ton Drive.

Une fois monté, tes fichiers seront accessibles au chemin :
```
/content/drive/MyDrive/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Vérifier que le montage a réussi
import os
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Google Drive monté dans /content/drive/MyDrive/")
else:
    print("❌ Impossible de monter Google Drive.")

Mounted at /content/drive
✅ Google Drive monté dans /content/drive/MyDrive/


---
# 📤 5. Obtenir le fichier audio à transcrire

Il existe **trois façons** de fournir l'audio :

### Option A : Téléverser un fichier depuis ton ordinateur
### Option B : Utiliser un fichier déjà présent sur Google Drive
### Option C : Télécharger l'audio d'une vidéo YouTube

> **Formats audio pris en charge par Whisper** : `.mp3`, `.wav`, `.flac`, `.ogg`, `.m4a`, `.wma`, `.aac` — et pratiquement tout format que `ffmpeg` peut lire (y compris les fichiers vidéo `.mp4`, `.mkv`, `.avi`, etc., dont Whisper extraira automatiquement la piste audio).

### 📤 Option A : Téléverser un fichier depuis ton ordinateur
Exécute la cellule suivante. Un bouton **« Choisir des fichiers »** apparaîtra pour téléverser ton audio.

In [ ]:
from google.colab import files

# Téléverser le(s) fichier(s) audio
uploaded = files.upload()

# Afficher les fichiers téléversés
for filename in uploaded.keys():
    print(f"✅ Fichier téléversé : {filename} ({len(uploaded[filename])/1e6:.1f} Mo)")

# Le fichier est disponible dans /content/nom_du_fichier
AUDIO_FILE = list(uploaded.keys())[0]  # On prend le premier
print(f"\n🎯 Fichier sélectionné pour la transcription : {AUDIO_FILE}")

Saving Pierrot-ConvergenceNumerique.mp3 to Pierrot-ConvergenceNumerique.mp3
✅ Fichier téléversé : Pierrot-ConvergenceNumerique.mp3 (122.7 Mo)

🎯 Fichier sélectionné pour la transcription : Pierrot-ConvergenceNumerique.mp3


### 📁 Option B : Utiliser un fichier de Google Drive
Si ton audio est dans Drive, il suffit d'indiquer le chemin complet. Par exemple :
```python
AUDIO_FILE = "/content/drive/MyDrive/mes_audios/entretien.mp3"
```

In [ ]:
# ✏️ MODIFIE ICI le chemin vers ton fichier dans Google Drive
# Exemple : AUDIO_FILE = "/content/drive/MyDrive/dossier/mon_audio.mp3"

AUDIO_FILE = "/content/drive/MyDrive/mon_audio.mp3"  # ← Changer ce chemin

import os
if os.path.exists(AUDIO_FILE):
    taille_mo = os.path.getsize(AUDIO_FILE) / 1e6
    print(f"✅ Fichier trouvé : {AUDIO_FILE} ({taille_mo:.1f} Mo)")
else:
    print(f"❌ Fichier introuvable : {AUDIO_FILE}")
    print("   Vérifiez le chemin. Vous pouvez explorer votre Drive dans le panneau de gauche (icône dossier).")

### 🎬 Option C : Télécharger l'audio d'une vidéo YouTube

Nous utilisons **yt-dlp**, l'outil le plus robuste pour télécharger du contenu depuis YouTube et d'autres plateformes. Nous extrairons **uniquement la piste audio** au format MP3 (plus léger que de télécharger toute la vidéo).

> ⚠️ **Avertissement légal** : le téléchargement de contenu YouTube peut être soumis à des restrictions de droits d'auteur. N'utilisez cette fonction qu'avec du contenu dont vous avez les droits ou l'autorisation.

In [ ]:
# ✏️ MODIFIE ICI : colle l'URL de la vidéo YouTube
YOUTUBE_URL = "https://www.youtube.com/watch?v=XXXXXXXXXX"  # ← Changer cette URL

# Nom du fichier de sortie (sans extension)
OUTPUT_NAME = "audio_youtube"

import subprocess

# Télécharger uniquement l'audio, le convertir en MP3 haute qualité
cmd = [
    "yt-dlp",
    "--extract-audio",           # Extraire uniquement l'audio
    "--audio-format", "mp3",     # Convertir en MP3
    "--audio-quality", "0",      # Qualité maximale (VBR ~245 kbps)
    "-o", f"/content/{OUTPUT_NAME}.%(ext)s",  # Chemin de sortie
    YOUTUBE_URL
]

print("⏳ Téléchargement de l'audio en cours...")
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    AUDIO_FILE = f"/content/{OUTPUT_NAME}.mp3"
    print(f"✅ Audio téléchargé : {AUDIO_FILE}")
    import os
    if os.path.exists(AUDIO_FILE):
        taille_mo = os.path.getsize(AUDIO_FILE) / 1e6
        print(f"   Taille : {taille_mo:.1f} Mo")
else:
    print("❌ Erreur lors du téléchargement :")
    print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)

---
# 🔊 6. Vérifier et écouter l'audio

Avant de lancer la transcription, il est conseillé de vérifier que le fichier a bien été chargé et de l'écouter brièvement :

In [ ]:
import subprocess, json
from IPython.display import Audio, display

# Obtenir les informations du fichier avec ffprobe
probe = subprocess.run(
    ["ffprobe", "-v", "quiet", "-print_format", "json", "-show_format", "-show_streams", AUDIO_FILE],
    capture_output=True, text=True
)

if probe.returncode == 0:
    info = json.loads(probe.stdout)
    fmt = info.get("format", {})
    duree_sec = float(fmt.get("duration", 0))
    duree_min = duree_sec / 60
    print(f"📄 Fichier  : {AUDIO_FILE}")
    print(f"⏱️  Durée    : {int(duree_min)}min {int(duree_sec % 60)}s ({duree_sec:.1f} secondes)")
    print(f"📦 Taille   : {float(fmt.get('size', 0))/1e6:.1f} Mo")
    print(f"🎵 Format   : {fmt.get('format_long_name', 'inconnu')}")
    for stream in info.get("streams", []):
        if stream.get("codec_type") == "audio":
            print(f"🔈 Codec    : {stream.get('codec_name', '?')} | "
                  f"Échantillonnage : {stream.get('sample_rate', '?')} Hz | "
                  f"Canaux : {stream.get('channels', '?')}")
    print()

# Lire l'audio directement dans le carnet
print("▶️  Écouter l'audio (aperçu) :")
display(Audio(AUDIO_FILE))

---
# ✍️ 7. Transcrire avec Whisper

Il existe **deux façons** d'utiliser Whisper :
- **Méthode A** — Ligne de commande (`!whisper ...`) : plus simple, idéale pour des transcriptions rapides.
- **Méthode B** — Script Python (`import whisper`) : plus flexible, permet de manipuler le résultat en code.

---

## Méthode A : Ligne de commande (recommandée pour commencer)

### Options principales

| Option | Description | Valeurs possibles |
|--------|-------------|-------------------|
| `--model` | Taille du modèle | `tiny`, `base`, `small`, `medium`, `large`, `turbo` |
| `--language` | Langue de l'audio (si connue) | `French`, `Spanish`, `English`, etc. ou code ISO (`fr`, `es`, `en`) |
| `--task` | Action à effectuer | `transcribe` (par défaut) ou `translate` (traduit en anglais) |
| `--output_format` | Format de sortie | `txt`, `srt`, `vtt`, `tsv`, `json`, `all` |
| `--output_dir` | Dossier de sortie | Chemin (ex. `/content/resultats`) |
| `--word_timestamps` | Horodatage par mot | `True` ou `False` |
| `--initial_prompt` | Contexte/vocabulaire pour guider la transcription | Texte libre |
| `--condition_on_previous_text` | Utiliser le texte précédent comme contexte | `True` (défaut) ou `False` |
| `--verbose` | Afficher la progression à l'écran | `True` ou `False` |

### À propos de `--initial_prompt`
Ce paramètre est **très utile** pour améliorer la transcription. Tu peux l'utiliser pour :
- Donner des **noms propres** ou des **termes techniques** que Whisper ne reconnaîtrait pas forcément.
- Indiquer le **style de ponctuation** souhaité.
- Exemple : `--initial_prompt "Entretien avec le Dr Dupont sur les récifs coralliens de Polynésie française."`

### À propos de `--condition_on_previous_text`
- En `True` (par défaut) : Whisper utilise la transcription précédente comme contexte. Cela améliore la cohérence, mais peut provoquer des **« hallucinations »** (texte répété ou inventé) à la fin d'audios longs.
- En `False` : chaque segment est transcrit indépendamment. Plus sûr pour les audios très longs.


### 📝 Exemple 1 : Transcription de base (qualité maximale)

In [ ]:
# Créer le dossier pour les résultats
!mkdir -p /content/resultats

# Transcription avec le modèle large (qualité maximale)
# Génère TOUS les formats de sortie (txt, srt, vtt, tsv, json)
!whisper "{AUDIO_FILE}" \
    --model large \
    --language French \
    --task transcribe \
    --output_format all \
    --output_dir /content/resultats \
    --verbose True

/usr/local/lib/python3.12/dist-packages/whisper/__init__.py:69: UserWarning: /root/.cache/whisper/large-v3.pt exists, but the SHA256 checksum does not match; re-downloading the file
  warnings.warn(
100%|██████████████████████████████████████| 2.88G/2.88G [00:19<00:00, 159MiB/s]
[00:00.000 --> 00:09.180]  Je te présente en une minute et après, il m'a donné l'autorisation d'une licence écrite pour l'interrompre si jamais, au point de vue de la logique, il fallait faire quelques répétitions.
[00:09.720 --> 00:22.100]  Donc Alain Thierot est un... Je faisais une remarque hier à des collègues en disant qu'il y a eu tant de startups sur le numérique dans les années 2000 et les jeunes autres de 22-23 ans nous refaisaient le monde.
[00:22.100 --> 00:45.100]  Et puis, j'ai constaté que les grands leaders et aujourd'hui, en plus tard, ceux qui influaient et qui nous aidaient le plus à penser le numérique, c'est-à-dire Tim Horry par exemple, pour ne prendre que lui, ou Bob Stein, c'était des gen

### 📝 Exemple 2 : Transcription rapide (modèle turbo)

In [ ]:
# Transcription rapide avec turbo (idéal pour les audios longs)
!whisper "{AUDIO_FILE}" \
    --model turbo \
    --language French \
    --output_format all \
    --output_dir /content/resultats \
    --verbose True

### 📝 Exemple 3 : Transcription avec vocabulaire spécifique et horodatage par mot

In [ ]:
# Transcription avec initial_prompt pour guider noms propres / vocabulaire
# et word_timestamps pour obtenir l'heure de chaque mot
!whisper "{AUDIO_FILE}" \
    --model large \
    --language French \
    --output_format all \
    --output_dir /content/resultats \
    --word_timestamps True \
    --initial_prompt "Conférence sur le patrimoine culturel immatériel avec Mme Lefebvre et l'architecte Jean-Pierre Moreau." \
    --verbose True

### 📝 Exemple 4 : Traduire un audio en anglais

In [ ]:
# Traduction en anglais (fonctionne uniquement avec medium ou large, PAS avec turbo)
!whisper "{AUDIO_FILE}" \
    --model large \
    --language French \
    --task translate \
    --output_format all \
    --output_dir /content/resultats \
    --verbose True

### 📝 Exemple 5 : Audios très longs ou avec « hallucinations »

Si Whisper commence à répéter du texte ou à inventer des mots à la fin d'un audio long, désactive `condition_on_previous_text` :

In [ ]:
# Pour les audios très longs ou avec des problèmes de répétition
!whisper "{AUDIO_FILE}" \
    --model large \
    --language French \
    --output_format all \
    --output_dir /content/resultats \
    --condition_on_previous_text False \
    --verbose True

---
## Méthode B : Script Python (plus de contrôle)

Cette méthode permet de manipuler directement le résultat de la transcription en Python. Utile si tu souhaites traiter la transcription ensuite (par exemple, nettoyer le texte, compter les mots, etc.).

In [ ]:
import whisper
import torch

# ===== CONFIGURATION (modifie selon tes besoins) =====
MODELE = "large"          # tiny | base | small | medium | large | turbo
LANGUE = "fr"             # fr, es, en, pt, de, it, ja, zh, etc. ou None pour détection auto
TACHE  = "transcribe"     # transcribe | translate (traduit en anglais)
PROMPT = None             # Texte de contexte, ex : "Réunion avec le professeur Martin sur la botanique."
HORODATAGE_MOT = False    # True pour obtenir les horodatages par mot
# =====================================================

print(f"⏳ Chargement du modèle '{MODELE}'...")
model = whisper.load_model(MODELE)
print(f"✅ Modèle chargé sur {'GPU' if next(model.parameters()).is_cuda else 'CPU'}")

print(f"\n⏳ Transcription en cours : {AUDIO_FILE}")
result = model.transcribe(
    AUDIO_FILE,
    language=LANGUE,
    task=TACHE,
    initial_prompt=PROMPT,
    word_timestamps=HORODATAGE_MOT,
    verbose=True
)

print("\n" + "="*60)
print("📝 TRANSCRIPTION COMPLÈTE :")
print("="*60)
print(result["text"])

### Enregistrer le résultat dans différents formats

Le résultat du script Python peut être exporté vers `.txt`, `.srt`, `.vtt`, etc., en utilisant les utilitaires intégrés de Whisper :

In [ ]:
from whisper.utils import get_writer
import os

# Dossier de sortie
OUTPUT_DIR = "/content/resultats"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Nom de base du fichier (sans extension)
audio_basename = os.path.splitext(os.path.basename(AUDIO_FILE))[0]

# Enregistrer dans tous les formats disponibles
for fmt in ["txt", "srt", "vtt", "tsv", "json"]:
    writer = get_writer(fmt, OUTPUT_DIR)
    writer(result, audio_basename)
    print(f"✅ Enregistré : {OUTPUT_DIR}/{audio_basename}.{fmt}")

print(f"\n📂 Tous les fichiers enregistrés dans : {OUTPUT_DIR}")

---
# 📄 8. Explorer et télécharger les résultats

### Formats de sortie

| Format | Extension | Description | Usage typique |
|--------|-----------|-------------|---------------|
| Texte brut | `.txt` | Texte seul, sans horodatage | Lecture, recherche, analyse de texte |
| SubRip | `.srt` | Sous-titres avec horodatage | Vidéos, VLC, YouTube, logiciels de montage |
| WebVTT | `.vtt` | Sous-titres web avec horodatage | Navigateurs, HTML5, plateformes web |
| TSV | `.tsv` | Données tabulaires (début, fin, texte) | Tableurs, analyse |
| JSON | `.json` | Données complètes et structurées | Programmation, traitement ultérieur |

In [ ]:
# Lister les fichiers générés
import os

OUTPUT_DIR = "/content/resultats"
print(f"📂 Fichiers dans {OUTPUT_DIR} :\n")
for f in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, f)
    taille = os.path.getsize(filepath) / 1024
    print(f"  📄 {f:40s} ({taille:.1f} Ko)")

### Lire la transcription en texte brut

In [ ]:
# Lire et afficher le fichier .txt
import glob

txt_files = glob.glob("/content/resultats/*.txt")
if txt_files:
    with open(txt_files[0], 'r', encoding='utf-8') as f:
        texte = f.read()
    print(f"📄 Fichier : {txt_files[0]}\n")
    print(texte[:3000])  # 3 000 premiers caractères
    if len(texte) > 3000:
        print(f"\n... [tronqué, total : {len(texte)} caractères]")
else:
    print("Aucun fichier .txt trouvé dans /content/resultats/")

### Lire les sous-titres (SRT)

In [ ]:
# Lire et afficher le fichier .srt (20 premières entrées)
import glob

srt_files = glob.glob("/content/resultats/*.srt")
if srt_files:
    with open(srt_files[0], 'r', encoding='utf-8') as f:
        lignes = f.readlines()
    print(f"📄 Fichier : {srt_files[0]}\n")
    print(''.join(lignes[:60]))  # ~20 sous-titres
    if len(lignes) > 60:
        print(f"\n... [tronqué, total : {len(lignes)} lignes]")
else:
    print("Aucun fichier .srt trouvé dans /content/resultats/")

### 💾 Télécharger les résultats

In [ ]:
from google.colab import files
import os

OUTPUT_DIR = "/content/resultats"

# Télécharger tous les fichiers générés
for f in sorted(os.listdir(OUTPUT_DIR)):
    filepath = os.path.join(OUTPUT_DIR, f)
    print(f"📥 Téléchargement : {f}")
    files.download(filepath)

### Ou copier les résultats vers Google Drive

In [ ]:
# Copier les résultats vers Google Drive
import shutil, os

# ✏️ MODIFIE le dossier de destination dans Drive
DRIVE_DEST = "/content/drive/MyDrive/Transcriptions_Whisper"

os.makedirs(DRIVE_DEST, exist_ok=True)

for f in os.listdir("/content/resultats"):
    src = os.path.join("/content/resultats", f)
    dst = os.path.join(DRIVE_DEST, f)
    shutil.copy2(src, dst)
    print(f"✅ Copié : {f} → {DRIVE_DEST}/")

print(f"\n📂 Tous les fichiers copiés dans : {DRIVE_DEST}")

---
# 💡 9. Conseils et résolution de problèmes

### Audio de mauvaise qualité
- Utilise le modèle `large` (plus robuste face au bruit de fond, aux accents, etc.).
- Spécifie `--language` plutôt que de laisser la détection automatique.
- Utilise `--initial_prompt` pour donner du contexte et du vocabulaire technique.

### Whisper répète du texte ou « hallucine »
- Utilise `--condition_on_previous_text False`.
- Réduis la température : dans le script Python, `temperature=0.0` produit des résultats plus stables.

### L'audio est très long (> 1 heure)
- `turbo` est beaucoup plus rapide que `large` (~8× plus rapide).
- Si la session Colab se déconnecte, rappelle-toi que les sessions gratuites ont une **limite d'environ 12 heures** et peuvent être interrompues en cas d'inactivité.
- Conseil : enregistre les résultats partiels dans Google Drive au fur et à mesure.

### Erreur de mémoire (CUDA out of memory)
- Utilise un modèle plus petit (`medium` ou `small`).
- Redémarre l'environnement d'exécution (menu *Environnement d'exécution → Redémarrer l'environnement d'exécution*) pour libérer la GPU.

### Langues prises en charge
Whisper prend en charge plus de **96 langues**. Voici les plus pertinentes :

| Code | Langue | Code | Langue |
|------|--------|------|--------|
| `fr` | Français | `en` | Anglais |
| `es` | Espagnol | `pt` | Portugais |
| `de` | Allemand | `it` | Italien |
| `ja` | Japonais | `zh` | Chinois |
| `ko` | Coréen | `ar` | Arabe |
| `nl` | Néerlandais | `ru` | Russe |
| `ca` | Catalan | `eu` | Basque |
| `oc` | Occitan | `br` | Breton |

Pour la liste complète : https://github.com/openai/whisper/blob/main/whisper/tokenizer.py

---
# 🚀 Référence rapide

```bash
# Transcription de base, qualité maximale
!whisper "mon_audio.mp3" --model large --language French --output_format all --output_dir /content/resultats

# Transcription rapide
!whisper "mon_audio.mp3" --model turbo --language French --output_format all --output_dir /content/resultats

# Avec vocabulaire spécifique
!whisper "mon_audio.mp3" --model large --language French --initial_prompt "Réunion avec le Dr Dupont" --output_format all --output_dir /content/resultats

# Traduire en anglais
!whisper "mon_audio.mp3" --model large --language French --task translate --output_format all --output_dir /content/resultats

# Télécharger l'audio depuis YouTube d'abord
!yt-dlp --extract-audio --audio-format mp3 --audio-quality 0 -o "/content/audio_yt.%(ext)s" "URL_DE_LA_VIDEO"
```

---
*Carnet de travail rédigé par le groupe d'intérêt sur l'acquisition assitée de textes du **consortium ARIANE** — Mars 2026*

*Basé sur [OpenAI Whisper](https://github.com/openai/whisper) (licence MIT)*